## Preprocess BactHeCom DB

In [1]:
import os
import pandas as pd
import numpy as np
import sqlite3
import requests
from datetime import datetime
from dateutil.relativedelta import relativedelta

from utils import *

import warnings
warnings.filterwarnings('ignore')

In [2]:
# paths
data_dir = "../data"
results_dir = "../results"
os.makedirs(results_dir, exist_ok=True)
bacthecom_db=f"{data_dir}/db_bacthecom.db"

In [3]:
## Load database 
conn = sqlite3.connect(bacthecom_db)

tbls = pd.read_sql_query("SELECT * FROM sqlite_master WHERE type='table';", con=conn)
tbls = tbls[tbls['name'] != 'sqlite_sequence']
print(f"Tables in the database: {tbls['name'].values}")

Tables in the database: ['paciente' 'factores_riesgo_infeccion_bmr' 'antibiograma'
 'semantic_mapping' 'episodio_infeccion' 'comorbilidad' 'signos_sintomas'
 'episodio_ingreso']


In [4]:
for tbl in tbls['name']:
    if tbl == "sqlite_sequence":
        continue    
    print(f"{tbl}")
    df = pd.read_sql_query(f"SELECT * FROM {tbl} LIMIT 5;", con=conn)
    print(f"Variables: {df.columns.values}")
    print("\n") 

paciente
Variables: ['record_id' 'sexo' 'fecha_nacimiento']


factores_riesgo_infeccion_bmr
Variables: ['record_id' 'fecha_ingreso' 'hospit_ano_previo' 'hospit_mes_previo'
 'hospit_ano_previo_uci' 'hemodialisis_permanente' 'dialisis_peritoneal'
 'cateter_venoso' 'sonda_urinaria' 'sonda_nasogastrica'
 'derivacion_ventriculoper' 'valvula_prot_cardiaca'
 'portador_otros_disposit']


antibiograma
Variables: ['episode_id' 'antimicrobiano' 'cmi' 'interpretacion']


semantic_mapping
Variables: ['table_name' 'variable_name' 'description' 'uri']


episodio_infeccion
Variables: ['episode_id' 'record_id' 'fecha_ingreso' 'fecha_cultivo' 'area_hosp'
 'id_cultivo' 'especimen' 'microorganismo' 'fenotipo_resistencia']


comorbilidad
Variables: ['record_id' 'fecha_ingreso' 'infarto' 'insuficiencia_cardiaca' 'evp'
 'e_cerebrovascular' 'demencia' 'e_pulmonar_cronica' 'ulcera_peptica'
 'colagenopatia' 'hemiplejia' 'erc' 'neoplasia_tratamiento_activo'
 'neoplasia_solida_metastasica' 'neoplasia_solida_no_me

## Preprocess and recode variables for ML modeling

### tbl_pacientes

In [5]:
tbl_pacientes = pd.read_sql_query("SELECT * FROM paciente;", con=conn)
tbl_pacientes.head()

,record_id,sexo,fecha_nacimiento
0,1,Hombre,1943-03-28
1,2,Mujer,1959-12-16
2,3,Hombre,1976-11-14
3,4,Hombre,1992-09-27
4,5,Mujer,1994-07-13


In [6]:
# Recode 'Hombre'/'Mujer' to 0/1
tbl_pacientes['sexo'] = tbl_pacientes['sexo'].map({'Hombre': 0, 'Mujer': 1})

### tbl_factores_riesgo_infeccion_bmr

In [7]:
tbl_factores_bmr = pd.read_sql_query("SELECT * FROM factores_riesgo_infeccion_bmr;", con=conn)
tbl_factores_bmr.head() 

,record_id,fecha_ingreso,hospit_ano_previo,hospit_mes_previo,hospit_ano_previo_uci,hemodialisis_permanente,dialisis_peritoneal,cateter_venoso,sonda_urinaria,sonda_nasogastrica,derivacion_ventriculoper,valvula_prot_cardiaca,portador_otros_disposit
0,1,2021-08-24,0,0,0,0,0,0,0,1,0,0,0
1,2,2023-06-18,0,0,0,0,0,0,0,1,0,0,0
2,3,2022-02-11,1,0,1,0,0,0,0,1,0,0,0
3,4,2021-05-15,0,0,0,0,0,0,0,1,0,0,0
4,5,2021-07-04,0,0,0,0,0,0,0,0,0,0,0


### tbl_episodios

In [8]:
tbl_episodios = pd.read_sql_query("SELECT * FROM episodio_ingreso;", con=conn)
tbl_episodios.fillna(0, inplace=True)
tbl_episodios.head()

,record_id,fecha_ingreso,fecha_alta,organo_aparato,foco_controlable,IRAs_nosocomial,mortalidad,uci_por_el_episodio,duracion_UCI,mujer_gestante,codigo_postal,paciente_residencia
0,1,2021-08-24,2021-09-09,0,0.0,Si,1,1,14,0,4007,0.0
1,2,2023-06-18,2023-07-13,0,0.0,Si,0,1,3,0,4740,0.0
2,3,2022-02-11,2022-04-06,Infeccion de cateter vascular,1.0,Si,0,1,24,0,4700,0.0
3,4,2021-05-15,2021-07-28,0,0.0,Si,0,1,42,0,18800,0.0
4,5,2021-07-04,2022-01-04,Infeccion tracto respiratorio inferior,0.0,Si,0,1,78,0,23009,0.0


In [9]:
# Recode IRAs_nosocomial to '0/1'
tbl_episodios['IRAs_nosocomial'] = tbl_episodios['IRAs_nosocomial'].map({'No': 0, 'Si': 1})

In [10]:
tbl_episodios['organo_aparato'].value_counts()

organo_aparato
0                                         2139
Infección de la vía urinaria superior      444
Fiebre sin foco                            275
Infeccion de cateter vascular              166
Infeccion vias biliares                    137
Infeccion intraabdominal                   127
Infeccion tracto respiratorio inferior      96
Infeccion de piel y partes blandas          78
Infeccion cardiovascular                    16
Infeccion osteoarticular                    11
Otros/Infeccion de etiologia incierta        9
Infeccion del SNC                            2
Infeccion genital                            1
Name: count, dtype: int64

In [11]:
# rename organo aparato values
infecname_to_infeccode = {
    'via_urinaria_superior': 'Infección de la vía urinaria superior',
    'fiebre_sin_foco': 'Fiebre sin foco',
    'cateter_vascular': 'Infeccion de cateter vascular',
    'vias_biliares': 'Infeccion vias biliares',
    'intraabdominal': 'Infeccion intraabdominal',
    'tracto_respiratorio_inferior': 'Infeccion tracto respiratorio inferior',
    'piel': 'Infeccion de piel y partes blandas',
    'cardiovascular': 'Infeccion cardiovascular',
    'osteoarticular': 'Infeccion osteoarticular',
    'etiologia_incierta': 'Otros/Infeccion de etiologia incierta',
    'snc': 'Infeccion del SNC',
    'genital': 'Infeccion genital'
}


In [12]:
# rename organo_aparato
tbl_episodios['organo_aparato'] = tbl_episodios['organo_aparato'].map({v: k for k, v in infecname_to_infeccode.items()})

# One-hot encode the 'organo_aparato' column
tbl_episodios_recoded = pd.get_dummies(tbl_episodios, columns=['organo_aparato'], prefix='infec',dtype=int)
tbl_episodios_recoded.head()


,record_id,fecha_ingreso,fecha_alta,foco_controlable,IRAs_nosocomial,mortalidad,uci_por_el_episodio,duracion_UCI,mujer_gestante,codigo_postal,...,infec_etiologia_incierta,infec_fiebre_sin_foco,infec_genital,infec_intraabdominal,infec_osteoarticular,infec_piel,infec_snc,infec_tracto_respiratorio_inferior,infec_via_urinaria_superior,infec_vias_biliares
0,1,2021-08-24,2021-09-09,0.0,1,1,1,14,0,4007,...,0,0,0,0,0,0,0,0,0,0
1,2,2023-06-18,2023-07-13,0.0,1,0,1,3,0,4740,...,0,0,0,0,0,0,0,0,0,0
2,3,2022-02-11,2022-04-06,1.0,1,0,1,24,0,4700,...,0,0,0,0,0,0,0,0,0,0
3,4,2021-05-15,2021-07-28,0.0,1,0,1,42,0,18800,...,0,0,0,0,0,0,0,0,0,0
4,5,2021-07-04,2022-01-04,0.0,1,0,1,78,0,23009,...,0,0,0,0,0,0,0,1,0,0


### tbl_comorbilidad
- Clusterizar cancer en clases y pivotar en columnas y codificar **0/1**

In [13]:
tbl_comorbilidades = pd.read_sql_query("SELECT * FROM comorbilidad;", con=conn)
tbl_comorbilidades.head()

,record_id,fecha_ingreso,infarto,insuficiencia_cardiaca,evp,e_cerebrovascular,demencia,e_pulmonar_cronica,ulcera_peptica,colagenopatia,...,diabetes_sin_lesion_organo_diana,diabetes_con_lesion_organo_diana,inmunosupresion,causa_inmunosupresion,fecha_TOS,TOS,fecha_TPH,TPH,clasificacion_quemadura,gran_quemado
0,1,2021-08-24,0,0,0,0,0,0,0,0,...,0,0,0,None,None,0,None,0,None,0
1,2,2023-06-18,0,0,0,0,0,0,0,0,...,0,0,0,None,None,0,None,0,None,0
2,3,2022-02-11,0,0,0,0,0,0,0,0,...,0,0,0,None,None,0,None,0,None,0
3,4,2021-05-15,0,0,0,0,0,0,0,0,...,0,0,0,None,None,0,None,0,"T20.29XA, T21.22XA, T21.24XA, T24.291A, T22.29...",1
4,5,2021-07-04,0,0,0,0,0,0,0,0,...,0,1,1,58606001,None,0,None,0,None,0


**Classify cancer codes in broad categories**

- Metadata extracted from https://www.icd10data.com/ICD10CM/Codes/C00-D49

In [14]:
tbl_comorbilidades_ = tbl_comorbilidades.copy()

tbl_comorbilidades_["tipo_cancer_list"] = (
    tbl_comorbilidades_["tipo_cancer"]episodio_infeccion
Variables: ['episode_id' 'record_id' 'fecha_ingreso' 'fecha_cultivo' 'area_hosp'
 'id_cultivo' 'especimen' 'microorganismo' 'fenotipo_resistencia']
        .dropna()
        .apply(lambda x: x.split(","))
)

# explode
tbl_comorbilidades_exploded = (
    tbl_comorbilidades_
        .assign(tipo_cancer_list=tbl_comorbilidades_["tipo_cancer_list"])
        .explode("tipo_cancer_list")
)
# recode tipo_cancer
tbl_comorbilidades_exploded["tipo_cancer_list"] = (
    tbl_comorbilidades_exploded["tipo_cancer_list"]
        .astype(str)
        .str.strip()
        .str.split(".")
        .str[0]
)
# classify using function defined in utils.py
tbl_comorbilidades_exploded['cancer_class'] = (
    tbl_comorbilidades_exploded['tipo_cancer_list']
        .map(classify_cancer)
)

# create dummies with cancer classes
dummies = (
    pd.get_dummies(tbl_comorbilidades_exploded['cancer_class'],
                   prefix='has_cancer',
                   dtype=int)
    .assign(record_id=tbl_comorbilidades_exploded['record_id'])
    .groupby("record_id")
    .max()
)

# merge w/ tbl_cmorbilidades
tbl_comorbilidades_recoded = tbl_comorbilidades.merge(
    dummies,
    on="record_id",
    how="left"
)

tbl_comorbilidades_recoded.head()


SyntaxError: invalid syntax. Perhaps you forgot a comma? (3976344939.py, line 4)

### tbl_signos

In [ ]:
tbl_signos = pd.read_sql_query("SELECT * FROM signos_sintomas;", con=conn)
tbl_signos.head()

,record_id,fecha_ingreso,foco,sepsis,shock_septico,qsofa,somnolencia_estupor_coma,situacion_funcional_basal,indice_de_charlson,escala_karnofsky,...,lesiones_piel,lesiones_mucosas,cefalea,dolores_articulares,temperatura,frec_cardiaca,frecuencia_respiratoria,tension_arterial_sist,tension_arterial_diast,saturacion_pO2
0,1,2021-08-24,None,1,1,0,None,None,11,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2,2023-06-18,None,0,0,0,None,None,4,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,3,2022-02-11,cateter venoso,0,0,0,None,None,2,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,4,2021-05-15,None,1,1,0,None,Necesita ayuda importante y asistencia médica ...,0,50.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,5,2021-07-04,pulmonar,0,0,0,None,"Normal, sin quejas ni evidencia de enfermedad",2,100.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
tbl_signos['foco'].unique()

array([None, 'cateter venoso', 'pulmonar', 'desconocido',
       'piel y partes blandas', 'urinario', 'intraabdominal',
       'osteoarticular', 'biliar', 'SNC', 'cardiovascular',
       'obstetricia/ginecológico', 'ORL', 'odontogeno'], dtype=object)

Rename and pivot:
- *foco*

Recode:
- *somnolencia_estupor_coma*
- *indice_de_charlson (low/medium/high)*
- *escala karnofsky (low/medium/high)*
- *hipotermia/hipertermia*
- *hipotension/hipertension*
- *hipoxemia*
- *taquipnea/taquicardia*

> Keep NaN values as they are

In [ ]:
# renaming and recoding 'foco'
tbl_signos['foco'] = tbl_signos['foco'].replace({'piel y partes blandas': 'piel', 'cateter venoso' : 'cateter'})
tbl_signos_recoded = pd.get_dummies(tbl_signos, columns=['foco'], prefix='foco', dtype=int)

# modify 'somnolencia_estupor_coma'
tbl_signos_recoded['somnolencia_estupor_coma'] = np.where(tbl_signos_recoded['somnolencia_estupor_coma'].isna(), np.nan,1)

# indice de charlson
tbl_signos_recoded['charlson_index_class'] = pd.cut(tbl_signos['indice_de_charlson'],bins=[-1, 3, 6, float('inf')],labels=['low', 'medium', 'high'])
tbl_signos_recoded = pd.get_dummies(tbl_signos_recoded, columns=['charlson_index_class'], prefix='charlson_index', dtype=int)

# escala karnofsky
# tbl_signos_recoded['karnofsky_class'] = pd.cut(tbl_signos['escala_karnofsky'],bins=[-1, 50, 80, float('inf')],labels=['low', 'medium', 'high'])
# tbl_signos_recoded = pd.get_dummies(tbl_signos_recoded, columns=['karnofsky_class'], prefix='karnofsky', dtype=int)

# hipo/hipertermia
tbl_signos_recoded['hipotermia'] = np.where(tbl_signos_recoded['temperatura'].isna(),np.nan,np.where(tbl_signos_recoded['temperatura'] <= 36, 1, 0))
tbl_signos_recoded['hipertermia'] = np.where(tbl_signos_recoded['temperatura'].isna(),np.nan,np.where(tbl_signos_recoded['temperatura'] >= 38, 1, 0))

# hipo/hipertensión
tbl_signos_recoded['hipotension'] = np.where((tbl_signos_recoded['tension_arterial_sist'] <= 90) &(tbl_signos_recoded['tension_arterial_diast'] <= 60),1, 0)
tbl_signos_recoded['hipertension'] = np.where((tbl_signos_recoded['tension_arterial_sist'] >= 140) &(tbl_signos_recoded['tension_arterial_diast'] >= 90),1, 0)

# taquipnea/taquicardia
tbl_signos_recoded['taquipnea'] = np.where(tbl_signos_recoded['frecuencia_respiratoria'].isna(),np.nan,np.where(tbl_signos_recoded['frecuencia_respiratoria'] > 20, 1, 0))
tbl_signos_recoded['taquicardia'] = np.where(tbl_signos_recoded['frec_cardiaca'].isna(),np.nan,np.where(tbl_signos_recoded['frec_cardiaca'] > 90, 1, 0))

# hipoxemia
tbl_signos_recoded['hipoxemia'] = np.where(tbl_signos_recoded['saturacion_pO2'].isna(),np.nan,np.where(tbl_signos_recoded['saturacion_pO2'] < 0.90, 1, 0))
tbl_signos_recoded.drop(columns=['indice_de_charlson', 'escala_karnofsky', 'temperatura', 'tension_arterial_sist',
                                  'tension_arterial_diast', 'frec_cardiaca', 'saturacion_pO2'], inplace=True)
tbl_signos_recoded.head()

,record_id,fecha_ingreso,sepsis,shock_septico,qsofa,somnolencia_estupor_coma,situacion_funcional_basal,barthel_inf_90,fiebre,tos,...,charlson_index_low,charlson_index_medium,charlson_index_high,hipotermia,hipertermia,hipotension,hipertension,taquipnea,taquicardia,hipoxemia
0,1,2021-08-24,1,1,0,NaN,None,NaN,0.0,0.0,...,0,0,1,NaN,NaN,0,0,NaN,NaN,NaN
1,2,2023-06-18,0,0,0,NaN,None,NaN,0.0,0.0,...,0,1,0,NaN,NaN,0,0,NaN,NaN,NaN
2,3,2022-02-11,0,0,0,NaN,None,NaN,0.0,0.0,...,1,0,0,NaN,NaN,0,0,NaN,NaN,NaN
3,4,2021-05-15,1,1,0,NaN,Necesita ayuda importante y asistencia médica ...,1.0,0.0,0.0,...,1,0,0,NaN,NaN,0,0,NaN,NaN,NaN
4,5,2021-07-04,0,0,0,NaN,"Normal, sin quejas ni evidencia de enfermedad",0.0,0.0,0.0,...,1,0,0,NaN,NaN,0,0,NaN,NaN,NaN


### tbl_microorganismo

In [ ]:
tbl_microorganismo = pd.read_sql_query("SELECT * FROM episodio_infeccion;", con=conn)
tbl_microorganismo = tbl_microorganismo.sort_values(by=["record_id","fecha_ingreso", "fecha_cultivo"])
tbl_microorganismo = tbl_microorganismo.drop(columns=['area_hosp','id_cultivo'])
tbl_microorganismo["fecha_cultivo"] = pd.to_datetime(tbl_microorganismo["fecha_cultivo"]).dt.date
tbl_microorganismo["fecha_ingreso"] = pd.to_datetime(tbl_microorganismo["fecha_ingreso"]).dt.date
tbl_microorganismo.head()

,episode_id,record_id,fecha_ingreso,fecha_cultivo,especimen,microorganismo,fenotipo_resistencia
0,1,1,2021-08-24,2021-08-31,Sangre,Pseudomonas aeruginosa,None
1,2,2,2023-06-18,2023-06-30,Sangre,Escherichia coli,None
2,3,3,2022-02-11,2021-04-23,Secreción bronquial (aspirado),Klebsiella pneumoniae ssp pneumoniae,BLEE
3,4,3,2022-02-11,2021-04-23,Secreción bronquial (aspirado),Pseudomonas aeruginosa,None
4,5,3,2022-02-11,2021-04-29,Secreción bronquial (aspirado),Pseudomonas aeruginosa,None


#### Recode tbl_microorganismos
- Create ***hemocultivo_si_no***
- Codify ***episodio_previo_si_no***
- How far back the previous episode occurs

In [ ]:
# create 'hemocultivo_si_no'
tbl_microorganismo["hemocultivo_si_no"] = np.where(
    tbl_microorganismo["especimen"] == "Sangre", 1, 0
)

# episodio previo codification (fecha_cultivo > fecha_ingreso)
tbl_microorganismo["episodio_previo_si_no"] = np.where(tbl_microorganismo["fecha_cultivo"] < tbl_microorganismo["fecha_ingreso"], 1, 0)

# get months diff
def get_months_diff (date1, date2):
    diff = relativedelta(date1,date2)
    total_months = diff.years * 12 + diff.months
    return abs(total_months)

# mask, only previous episodes
mask = tbl_microorganismo["episodio_previo_si_no"] == 1

tbl_microorganismo.loc[mask, "pisodio_previo_months_diff"] = (tbl_microorganismo.loc[mask]
    .apply(lambda r: get_months_diff(r["fecha_cultivo"], r["fecha_ingreso"]), axis=1))

# code if recent previous episode
# tbl_microorganismo["6m_within_episodio_previo"] = np.where(tbl_microorganismo["months_diff_episodio_previo"] < 6, 1,0)
# tbl_microorganismo["3m_within_episodio_previo"] = np.where(tbl_microorganismo["months_diff_episodio_previo"] < 3, 1,0)
# tbl_microorganismo["1m_within_episodio_previo"] = np.where(tbl_microorganismo["months_diff_episodio_previo"] < 1, 1,0)

tbl_microorganismo.head()

,episode_id,record_id,fecha_ingreso,fecha_cultivo,especimen,microorganismo,fenotipo_resistencia,hemocultivo_si_no,episodio_previo_si_no,months_diff_episodio_previo
0,1,1,2021-08-24,2021-08-31,Sangre,Pseudomonas aeruginosa,None,1,0,NaN
1,2,2,2023-06-18,2023-06-30,Sangre,Escherichia coli,None,1,0,NaN
2,3,3,2022-02-11,2021-04-23,Secreción bronquial (aspirado),Klebsiella pneumoniae ssp pneumoniae,BLEE,0,1,9.0
3,4,3,2022-02-11,2021-04-23,Secreción bronquial (aspirado),Pseudomonas aeruginosa,None,0,1,9.0
4,5,3,2022-02-11,2021-04-29,Secreción bronquial (aspirado),Pseudomonas aeruginosa,None,0,1,9.0


### Classify microorganismos into groups
- Bacterial groups: ***ECOLI, KP, PSA, SA, OEB, NOEB***

In [ ]:
tbl_microorganismo_recoded = tbl_microorganismo.copy()

# recode microorganism
tbl_microorganismo_recoded["microorganismo_recoded"] = (
    tbl_microorganismo_recoded["microorganismo"]
    .apply(lambda x: '_'.join(x.split(' ')[:2]))
)
# classify microorganism
tbl_microorganismo_recoded["microorganismo_class"] = tbl_microorganismo_recoded["microorganismo_recoded"].map(classify_microorganism)

# create dummies
# dummies = (pd.get_dummies(tbl_microorganismo_recoded["microorganismo_class"], prefix="microorganismo", dtype=int)
#            .assign(record_id=tbl_microorganismo_recoded["record_id"])
#            .assign(fecha_cultivo=tbl_microorganismo_recoded["fecha_cultivo"])
#            .groupby(by=['record_id', 'fecha_cultivo'])
#            .max()
# )

# # tbl_microorganismo_recoded = (pd.merge(left=tbl_microorganismo_recoded, right=dummies, on=["record_id","fecha_cultivo"])
# #                                 .drop(columns=['microorganismo','microorganismo_class'])).drop_duplicates()
# tbl_microorganismo_recoded = (pd.merge(left=tbl_microorganismo_recoded, right=dummies, on=["record_id","fecha_cultivo"])
#                                 .drop_duplicates())

# one-hot encoding for resistance
tbl_microorganismo_recoded["has_BMR_resistance"] = np.where(tbl_microorganismo_recoded["fenotipo_resistencia"].isna(), 0,1)
#tbl_microorganismo_recoded = tbl_microorganismo_recoded.drop(columns=['fenotipo_resistencia'])
tbl_microorganismo_recoded.head()

,episode_id,record_id,fecha_ingreso,fecha_cultivo,especimen,microorganismo,fenotipo_resistencia,hemocultivo_si_no,episodio_previo_si_no,months_diff_episodio_previo,microorganismo_recoded,microorganismo_class,has_BMR_resistance
0,1,1,2021-08-24,2021-08-31,Sangre,Pseudomonas aeruginosa,None,1,0,NaN,Pseudomonas_aeruginosa,PSA,0
1,2,2,2023-06-18,2023-06-30,Sangre,Escherichia coli,None,1,0,NaN,Escherichia_coli,ECOLI,0
2,3,3,2022-02-11,2021-04-23,Secreción bronquial (aspirado),Klebsiella pneumoniae ssp pneumoniae,BLEE,0,1,9.0,Klebsiella_pneumoniae,KP,1
3,4,3,2022-02-11,2021-04-23,Secreción bronquial (aspirado),Pseudomonas aeruginosa,None,0,1,9.0,Pseudomonas_aeruginosa,PSA,0
4,5,3,2022-02-11,2021-04-29,Secreción bronquial (aspirado),Pseudomonas aeruginosa,None,0,1,9.0,Pseudomonas_aeruginosa,PSA,0


In [ ]:
tbl_microorganismo_recoded[tbl_microorganismo_recoded["record_id"]==34]

,episode_id,record_id,fecha_ingreso,fecha_cultivo,especimen,microorganismo,fenotipo_resistencia,hemocultivo_si_no,episodio_previo_si_no,months_diff_episodio_previo,microorganismo_recoded,microorganismo_class,has_BMR_resistance
88,89,34,2021-08-08,2021-08-12,Exudado herida,Pseudomonas aeruginosa,None,0,0,NaN,Pseudomonas_aeruginosa,PSA,0
89,90,34,2021-08-08,2021-08-15,Punta de catéter (genérico),Pseudomonas aeruginosa,None,0,0,NaN,Pseudomonas_aeruginosa,PSA,0
90,91,34,2021-08-08,2021-08-19,Quemadura (escara),Pseudomonas aeruginosa,None,0,0,NaN,Pseudomonas_aeruginosa,PSA,0
91,92,34,2021-08-08,2021-08-20,Secreción bronquial (aspirado),Pseudomonas aeruginosa,None,0,0,NaN,Pseudomonas_aeruginosa,PSA,0
92,93,34,2021-08-08,2021-08-20,Secreción bronquial (aspirado),Stenotrophomonas maltophilia,None,0,0,NaN,Stenotrophomonas_maltophilia,NOEB,0
93,94,34,2021-08-08,2021-08-22,Punta de catéter (genérico),Pseudomonas aeruginosa,None,0,0,NaN,Pseudomonas_aeruginosa,PSA,0
94,95,34,2021-08-08,2021-08-22,Punta de catéter (genérico),Stenotrophomonas maltophilia,None,0,0,NaN,Stenotrophomonas_maltophilia,NOEB,0
95,96,34,2021-08-08,2021-08-22,Sangre,Pseudomonas aeruginosa,None,1,0,NaN,Pseudomonas_aeruginosa,PSA,0
96,97,34,2021-08-08,2021-08-22,Sangre,Pseudomonas aeruginosa MR,MR,1,0,NaN,Pseudomonas_aeruginosa,PSA,1
97,98,34,2021-08-08,2021-08-22,Punta de catéter (genérico),Pseudomonas aeruginosa,None,0,0,NaN,Pseudomonas_aeruginosa,PSA,0


### Construct matrix grouping by ['record_id','fecha_ingreso']

In [ ]:
max_prev = (
    tbl_microorganismo_recoded
    .groupby(["record_id","fecha_ingreso"])["episodio_previo_si_no"]
    .sum()
    .max()
)
print(max_prev)

7


In [ ]:
max_non_prev = (
    (tbl_microorganismo_recoded["episodio_previo_si_no"] == 0)
    .groupby([tbl_microorganismo_recoded["record_id"],
              tbl_microorganismo_recoded["fecha_ingreso"]])
    .sum()     # counts True values = count of non previous episodes
    .reset_index(name='otros_cultivos')
)

max_non_prev

,record_id,fecha_ingreso,otros_cultivos
0,1,2021-08-24,1
1,2,2023-06-18,1
2,3,2022-02-11,4
3,4,2021-05-15,6
4,5,2021-07-04,7
...,...,...,...
3402,3045,2023-06-21,4
3403,3046,2023-08-02,4
3404,3047,2023-08-30,2
3405,3048,2023-09-25,1


In [ ]:
tbl_microorganismo_recoded[(tbl_microorganismo_recoded['hemocultivo_si_no']==1)&(tbl_microorganismo_recoded["episodio_previo_si_no"]==0)]["record_id"].value_counts()

record_id
1774    12
88      10
596     10
167      9
34       9
        ..
1119     1
1120     1
1121     1
1124     1
1110     1
Name: count, Length: 2802, dtype: int64

### collapse 'record_id' into columns 
- Pivotar info de episodios en columnas: ***cultivo_previo_microorganismo_1, ..., cultivo_previo_microorganismo_n***

In [ ]:
blood_per_episode = (
    tbl_microorganismo_recoded
    .groupby(["record_id", "fecha_ingreso"])["hemocultivo_si_no"]
    .max()      # True if ANY row in the group has sangre
    .reset_index(name="has_sangre")
)


In [ ]:
blood_per_episode[blood_per_episode["has_sangre"] == False]

,record_id,fecha_ingreso,has_sangre


In [ ]:
blood_per_episode

,record_id,fecha_ingreso,has_sangre
0,1,2021-08-24,1
1,2,2023-06-18,1
2,3,2022-02-11,1
3,4,2021-05-15,1
4,5,2021-07-04,1
...,...,...,...
3402,3045,2023-06-21,1
3403,3046,2023-08-02,1
3404,3047,2023-08-30,1
3405,3048,2023-09-25,1


In [ ]:
count_no_blood

NameError: name 'count_no_blood' is not defined

In [ ]:
prev_episodes = tbl_microorganismo_recoded[tbl_microorganismo_recoded["episodio_previo_si_no"]==1].copy()

# sort again, robust order
prev_episodes = prev_episodes.sort_values(["record_id", "fecha_ingreso", "fecha_cultivo"])
# For each patient + fecha_ingreso, enumerate previous microorganisms
prev_episodes["micro_index"] = prev_episodes.groupby(["record_id", "fecha_ingreso"]).cumcount() + 1
pivot_prev = prev_episodes.pivot_table(
    index=["record_id", "fecha_ingreso"], 
    columns="micro_index", 
    values="microorganismo_recoded",
    aggfunc="first"
).reset_index()
pivot_prev = pivot_prev.rename(columns=lambda x: f"cultivo_previo_microorganism_{x}" if isinstance(x,int) else x)
tbl_collapsed = tbl_microorganismo.merge(
    pivot_prev,
    on=["record_id", "fecha_ingreso"],
    how="left"
)
tbl_collapsed.
tbl_collapsed.head()

,episode_id,record_id,fecha_ingreso,fecha_cultivo,area_hosp,id_cultivo,especimen,microorganismo,fenotipo_resistencia,hemocultivo_si_no,...,6m_within_episodio_previo,3m_within_episodio_previo,1m_within_episodio_previo,cultivo_previo_microorganism_1,cultivo_previo_microorganism_2,cultivo_previo_microorganism_3,cultivo_previo_microorganism_4,cultivo_previo_microorganism_5,cultivo_previo_microorganism_6,cultivo_previo_microorganism_7
0,1,1,2021-08-24,2021-08-31,UCI,102007829,Sangre,Pseudomonas aeruginosa,None,1,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,2,2023-06-18,2023-06-30,General,103800271,Sangre,Escherichia coli,None,1,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,3,2022-02-11,2021-04-23,UCI,None,Secreción bronquial (aspirado),Klebsiella pneumoniae ssp pneumoniae,BLEE,0,...,1,0,0,Klebsiella_pneumoniae,Pseudomonas_aeruginosa,Pseudomonas_aeruginosa,NaN,NaN,NaN,NaN
3,4,3,2022-02-11,2021-04-23,UCI,None,Secreción bronquial (aspirado),Pseudomonas aeruginosa,None,0,...,1,0,0,Klebsiella_pneumoniae,Pseudomonas_aeruginosa,Pseudomonas_aeruginosa,NaN,NaN,NaN,NaN
4,5,3,2022-02-11,2021-04-29,UCI,None,Secreción bronquial (aspirado),Pseudomonas aeruginosa,None,0,...,1,0,0,Klebsiella_pneumoniae,Pseudomonas_aeruginosa,Pseudomonas_aeruginosa,NaN,NaN,NaN,NaN


### tbl_antibiograma

In [15]:
tbl_antibiograma = pd.read_sql_query("SELECT * FROM antibiograma", con=conn)
tbl_antibiograma.head()

,episode_id,antimicrobiano,cmi,interpretacion
0,1,Cefazolina,>16 mg/L,R
1,1,Trimetoprim,>4 mg/L,R
2,1,Ertapenem,>1 mg/L,R
3,1,Cefuroxima,>8 mg/L,R
4,1,Trimetroprim/sulfametoxazol,>4/76 mg/L,R


In [20]:
tbl_antibiograma[tbl_antibiograma['antimicrobiano']=='Meropenem']

,episode_id,antimicrobiano,cmi,interpretacion
104,11,Meropenem,16 mg/L,R
119,12,Meropenem,>32 mg/L,R
125,13,Meropenem,None,R
147,14,Meropenem,16 mg/L,R
227,21,Meropenem,32 mg/L,R
...,...,...,...,...
151577,7278,Meropenem,1 mg/L,S
151579,7279,Meropenem,1 mg/L,S
151593,7280,Meropenem,<=0.12 mg/L,S
151609,7281,Meropenem,<=0.12 mg/L,S
